In [0]:
%run ./01_config

In [0]:
"""
06_gold_survival.py  —  Gold layer: analysis-ready tables

Part of the SAP PM cost-aware maintenance optimisation pipeline.
Run order is filename order: 01 through 12.
"""

# 06 — Gold: analysis-ready tables

# Four outputs, one per downstream consumer:

# Table  -  Feeds
# gold_survival_intervals  -  Prediction layer (Section 4.2) — lifelines / scikit-survival / XGBoost-AFT
# gold_class_cost_params  -  Decision layer (Section 4.3.1) — cost-rate objective
# gold_incumbent_cycles  -  Baseline policy (Section 5.1) — as-is MPLA cycles
# gold_escalation_training  -  Decision layer (Section 4.3.2) — cost-of-delay ranking

# The interval construction is the part worth reading closely: it is where the censoring
# decision described in Section 4.2.1 is actually implemented.

# Shared configuration from '01_config' is assumed to be in scope.

from pyspark.sql import functions as F
use_project_schema()

ORDER_EXPR = ("PARTITION BY equipment_id ORDER BY event_date, "
              "CASE event_type WHEN 'INSTALL' THEN 0 WHEN 'PM' THEN 1 ELSE 2 END")

# Survival intervals

# Each interval opens at a renewal point (commissioning, completed corrective repair, or
# preventive intervention) and closes at the next event. It is an observed event only
# when the next event is a breakdown; a PM-terminated interval is right-censored, not
# discarded, because it carries the information that the item survived to that age.
# Intervals still open at the end of the window are censored at OBS_END.

# Tie-breaking puts PM before FAILURE on the same date: a same-day pair is read as
# "maintained, then failed", which censors conservatively rather than crediting the PM
# with a survival it did not deliver.

spark.sql(f"""
CREATE OR REPLACE TABLE {tbl('gold_survival_intervals')} AS
WITH eq AS (
    SELECT e.equipment_id,
           e.equipment_class,
           e.iso14224_group,
           e.start_up_date,
           e.planning_plant AS plant,
           f.criticality
    FROM {tbl('silver_equipment')} e
    LEFT JOIN {tbl('silver_functional_location')} f
           ON e.functional_location_id = f.functional_location_id
),
ev AS (
    SELECT equipment_id, start_up_date AS event_date, 'INSTALL' AS event_type FROM eq
    UNION ALL
    -- Intrinsic failures only. Escalation-driven breakdowns (originating_notification_id
    -- IS NOT NULL) come from the independent defect overlay, which does not touch virtual
    -- age; including them would contaminate the Weibull recovery with a second, unrelated
    -- failure mechanism. They are modelled separately by the prioritizer instead.
    -- On v2 data the linkage is always NULL, so this predicate is a no-op there.
    SELECT equipment_id, COALESCE(malfunction_start, notification_date), 'FAILURE'
    FROM {tbl('silver_notification')}
    WHERE notification_type = 'M1' AND originating_notification_id IS NULL
    UNION ALL
    SELECT equipment_id, basic_start, 'PM'
    FROM {tbl('silver_order')} WHERE order_class = 'PREVENTIVE'
),
seq AS (
    SELECT equipment_id, event_date, event_type,
           ROW_NUMBER() OVER ({ORDER_EXPR})                                   AS seq_no,
           LEAD(event_date) OVER ({ORDER_EXPR})                               AS next_event_date,
           LEAD(event_type) OVER ({ORDER_EXPR})                               AS next_event_type,
           SUM(CASE WHEN event_type = 'FAILURE' THEN 1 ELSE 0 END)
               OVER ({ORDER_EXPR} ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
                                                                              AS failures_to_date,
           MAX(CASE WHEN event_type = 'PM' THEN event_date END)
               OVER ({ORDER_EXPR} ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
                                                                              AS last_pm_date
    FROM ev
    WHERE event_date IS NOT NULL AND event_date <= DATE'{OBS_END}'
)
SELECT
    md5(concat_ws('|', s.equipment_id, CAST(s.seq_no AS STRING)))             AS interval_id,
    s.equipment_id,
    eq.equipment_class,
    eq.iso14224_group,
    eq.plant,
    eq.criticality,
    s.seq_no,
    s.event_type                                                              AS opened_by,
    s.event_date                                                              AS interval_start,
    COALESCE(s.next_event_date, DATE'{OBS_END}')                              AS interval_end,
    DATEDIFF(COALESCE(s.next_event_date, DATE'{OBS_END}'), s.event_date)      AS duration_days,
    CASE WHEN s.next_event_type = 'FAILURE' THEN 1 ELSE 0 END                 AS event_observed,
    CASE WHEN s.next_event_type = 'FAILURE' THEN 'FAILURE'
         WHEN s.next_event_type = 'PM'      THEN 'CENSORED_PM'
         ELSE                                    'CENSORED_WINDOW' END        AS termination,
    DATEDIFF(s.event_date, eq.start_up_date)                                  AS age_at_start_days,
    s.failures_to_date - CASE WHEN s.event_type = 'FAILURE' THEN 1 ELSE 0 END AS prior_failure_count,
    DATEDIFF(s.event_date, s.last_pm_date)                                    AS days_since_last_pm,
    CASE WHEN s.event_date <= DATE'{TRAIN_CUTOFF}' THEN 'train' ELSE 'test' END AS split
FROM seq s
JOIN eq ON s.equipment_id = eq.equipment_id
WHERE DATEDIFF(COALESCE(s.next_event_date, DATE'{OBS_END}'), s.event_date) > 0
""")

iv = spark.table(tbl("gold_survival_intervals"))
print(f"gold_survival_intervals: {iv.count():,} intervals")
display(iv.groupBy("split", "termination").count().orderBy("split", "termination"))

# Per-class summary. Watch the censoring rate: classes where almost every interval is
# PM-censored carry very little failure signal, and their Weibull fits in Section 4.2.3 will be
# correspondingly wide. Worth reporting alongside the recovery errors rather than after
# someone asks.

class_summary = (
    iv.groupBy("equipment_class")
      .agg(F.count("*").alias("intervals"),
           F.sum("event_observed").alias("observed_failures"),
           F.round(1 - F.avg("event_observed"), 3).alias("censoring_rate"),
           F.round(F.avg("duration_days"), 1).alias("mean_duration_days"),
           F.countDistinct("equipment_id").alias("equipment_items"))
      .orderBy("equipment_class")
)
display(class_summary)

# Class cost parameters

# The empirical breakdown-to-preventive ratio that drives the cost-rate objective. Medians,
# not means — settled costs are right-skewed and a handful of expensive breakdowns would
# otherwise set the ratio for the whole class.

spark.sql(f"""
CREATE OR REPLACE TABLE {tbl('gold_class_cost_params')} AS
SELECT
    e.equipment_class,
    COUNT(*)                                                                  AS orders,
    SUM(CASE WHEN o.order_class = 'PREVENTIVE' THEN 1 ELSE 0 END)             AS preventive_orders,
    SUM(CASE WHEN o.order_class = 'CORRECTIVE' THEN 1 ELSE 0 END)             AS corrective_orders,
    ROUND(PERCENTILE_APPROX(CASE WHEN o.order_class = 'PREVENTIVE'
                                 THEN o.total_actual_cost END, 0.5), 2)       AS preventive_cost_median,
    ROUND(PERCENTILE_APPROX(CASE WHEN o.order_class = 'CORRECTIVE'
                                 THEN o.total_actual_cost END, 0.5), 2)       AS breakdown_cost_median,
    ROUND(PERCENTILE_APPROX(CASE WHEN o.order_class = 'CORRECTIVE'
                                 THEN o.total_actual_cost END, 0.5)
        / NULLIF(PERCENTILE_APPROX(CASE WHEN o.order_class = 'PREVENTIVE'
                                 THEN o.total_actual_cost END, 0.5), 0), 3)   AS cost_ratio_empirical,
    ROUND(AVG(CASE WHEN o.order_class = 'PREVENTIVE' THEN c.actual_work_hours END), 2) AS preventive_hours_avg,
    ROUND(AVG(CASE WHEN o.order_class = 'CORRECTIVE' THEN c.actual_work_hours END), 2) AS corrective_hours_avg
FROM {tbl('silver_order')} o
JOIN {tbl('silver_equipment')} e ON o.equipment_id = e.equipment_id
LEFT JOIN (SELECT order_id, SUM(actual_work_hours) AS actual_work_hours
           FROM {tbl('silver_confirmation')} GROUP BY order_id) c
       ON o.order_id = c.order_id
GROUP BY e.equipment_class
ORDER BY e.equipment_class
""")

display(spark.table(tbl("gold_class_cost_params")))

# Incumbent calendar cycles — the baseline policy of Section 5.1

spark.sql(f"""
CREATE OR REPLACE TABLE {tbl('gold_incumbent_cycles')} AS
SELECT e.equipment_class,
       COUNT(*)                                   AS plans,
       MIN(m.cycle_length)                        AS cycle_days_min,
       ROUND(AVG(m.cycle_length), 1)              AS cycle_days_mean,
       MAX(m.cycle_length)                        AS cycle_days_max,
       FIRST(m.cycle_unit)                        AS cycle_unit
FROM {tbl('silver_maintenance_plan')} m
JOIN {tbl('silver_equipment')} e ON m.equipment_id = e.equipment_id
GROUP BY e.equipment_class
ORDER BY e.equipment_class
""")

display(spark.table(tbl("gold_incumbent_cycles")))

# Escalation training set (FR4 / RQ3)

# One row per defect notification (M2), labelled by whether a breakdown notification cites
# it as its origin. Features are restricted to what is knowable at notification time — no
# leakage from the outcome window.

# Skipped with a warning if the extract has no escalation linkage (i.e. on v2 data).

has_link = (spark.table(tbl("silver_notification"))
                 .filter(F.col("originating_notification_id").isNotNull()).limit(1).count() > 0)

if has_link:
    spark.sql(f"""
    CREATE OR REPLACE TABLE {tbl('gold_escalation_training')} AS
    WITH defects AS (
        SELECT n.notification_id, n.equipment_id, n.notification_date, n.priority,
               n.damage_code, n.cause_code
        FROM {tbl('silver_notification')} n
        WHERE n.notification_type = 'M2'
    ),
    escalated AS (
        SELECT DISTINCT originating_notification_id AS notification_id,
               MIN(malfunction_start) AS escalation_date
        FROM {tbl('silver_notification')}
        WHERE notification_type = 'M1' AND originating_notification_id IS NOT NULL
        GROUP BY originating_notification_id
    )
    SELECT d.notification_id,
           d.equipment_id,
           e.equipment_class,
           e.iso14224_group,
           f.criticality,
           d.notification_date,
           d.priority,
           d.damage_code,
           d.cause_code,
           DATEDIFF(d.notification_date, e.start_up_date)              AS equipment_age_days,
           (SELECT COUNT(*) FROM {tbl('silver_notification')} p
             WHERE p.equipment_id = d.equipment_id
               AND p.notification_type = 'M1'
               AND p.malfunction_start < d.notification_date)          AS prior_failures,
           CASE WHEN x.notification_id IS NOT NULL THEN 1 ELSE 0 END   AS escalated,
           DATEDIFF(x.escalation_date, d.notification_date)            AS days_to_escalation,
           CASE WHEN d.notification_date <= DATE'{TRAIN_CUTOFF}'
                THEN 'train' ELSE 'test' END                           AS split
    FROM defects d
    JOIN {tbl('silver_equipment')} e ON d.equipment_id = e.equipment_id
    LEFT JOIN {tbl('silver_functional_location')} f
           ON e.functional_location_id = f.functional_location_id
    LEFT JOIN escalated x ON d.notification_id = x.notification_id
    """)
    esc = spark.table(tbl("gold_escalation_training"))
    print(f"gold_escalation_training: {esc.count():,} defect notifications")
    display(esc.groupBy("split", "escalated").count().orderBy("split", "escalated"))
else:
    print("SKIPPED: no originating_notification_id in this extract. The escalation mechanism "
          "(Section 4.1.1) and therefore FR4 / RQ3 require v2.1 data. Re-run this notebook after "
          "regenerating with the escalation overlay enabled.")

# Table comments

# Documented in the catalog rather than only in the report, so the schema is
# self-describing for the examiner and for anyone reusing the published benchmark.

comments = {
    "gold_survival_intervals": "One row per at-risk interval. event_observed=1 for breakdown, 0 for right-censored (PM or window end). Feeds RQ1 survival models (report section 3.3).",
    "gold_class_cost_params": "Per-class empirical breakdown-to-preventive cost ratio and effort, input to the cost-rate objective (report section 3.4.1).",
    "gold_incumbent_cycles": "As-is MPLA calendar cycles per class - the conventional-practice baseline (report section 3.5.1).",
    "gold_escalation_training": "Defect notifications labelled by escalation to breakdown; features restricted to notification time. Feeds RQ3 cost-of-delay ranking (report section 3.4.2).",
}
for name, comment in comments.items():
    if spark.catalog.tableExists(tbl(name)):
        spark.sql(f"COMMENT ON TABLE {tbl(name)} IS '{comment}'")
        print(f"documented {name}")